# Importing Libraries

In [16]:
import numpy as np 
import pandas as pd 

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load Dataset

In [17]:
matches = pd.read_csv("../data/processed/matches.csv")

In [18]:
matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 23500 entries, 0 to 23499
Data columns (total 26 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   date                        23500 non-null  str    
 1   home_team                   23500 non-null  str    
 2   away_team                   23500 non-null  str    
 3   home_score                  23500 non-null  int64  
 4   away_score                  23500 non-null  int64  
 5   goal_diff                   23500 non-null  int64  
 6   tournament                  23500 non-null  str    
 7   country                     23500 non-null  str    
 8   neutral                     23500 non-null  bool   
 9   home_is_host                23500 non-null  int64  
 10  away_is_host                23500 non-null  int64  
 11  elo_diff                    23500 non-null  float64
 12  home_higher_elo             23500 non-null  int64  
 13  home_recent_win_rate        23500 non-null

# Preparation

## Define features and target

In [19]:
feature_cols = [
    "neutral",
    "home_is_host",
    "away_is_host",
    "elo_diff",
    "recent_form_diff",
    "diff_in_avg_goals",
    "diff_in_total_conceded",
    "tournament_weight"
]

target = "match_result"

X = matches[feature_cols]
y = matches[target]

# Train-test Split

In [20]:
matches["date"] = pd.to_datetime(matches["date"])
split_date = "2022-12-19"
train = matches[matches["date"] < split_date]
test = matches[matches["date"] >= split_date]

# Checking train-test split ratio
print(len(train)/len(matches)*100)

86.01702127659574


In [21]:
# Preparing train and test sets
X_train = train[feature_cols]
y_train = train[target]
X_test = test[feature_cols]
y_test = test[target]

print(f"Training set: {len(train)} matches")
print(f"Test set: {len(test)} matches")
print(f"\nTarget distribution:\n{y_train.value_counts(normalize=True)}")

Training set: 20214 matches
Test set: 3286 matches

Target distribution:
match_result
home_win    0.483427
away_win    0.279954
draw        0.236618
Name: proportion, dtype: float64


## Encoding

In [22]:
# Encode target labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)  
print(f"\nLabel encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")


Label encoding: {'away_win': 0, 'draw': 1, 'home_win': 2}


# Model Training

## Logistic Regression

Logistic Regression serves as the baseline model for this prediction project.

In [23]:
# Standardise the training and test data
std_x = StandardScaler()
X_train = std_x.fit_transform(X_train)
X_test = std_x.transform(X_test)

In [28]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
print("\n--- Logistic Regression ---")
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print(classification_report(y_test, lr_preds))


--- Logistic Regression ---
Accuracy: 0.5654
              precision    recall  f1-score   support

    away_win       0.56      0.65      0.60       974
        draw       0.29      0.25      0.26       774
    home_win       0.70      0.67      0.68      1538

    accuracy                           0.57      3286
   macro avg       0.51      0.52      0.52      3286
weighted avg       0.56      0.57      0.56      3286

